# PE6201 Final Project — Final Evaluation v3 (No Colab Secrets)

This version **does not use Colab Secrets**.

When you run the setup cell, Colab will show a **hidden password-style input box**. Paste your OpenRouter API key there. The key is kept only in the current runtime memory, is not printed, and is not saved into this notebook.

This notebook will:
1. reproduce the 48/27 split, majority baseline and TF-IDF score;
2. run the 10 fixed end-to-end advice cases through OpenRouter;
3. automatically report route correctness, evidence recall@3, token use and abstention;
4. save the outputs to `advice_eval_outputs.csv`;
5. let **you hand-mark** specificity, note support and no unsupported invention;
6. save `advice_eval_final.csv` and `final_eval_summary.json`.

**Security:** use a newly generated OpenRouter key if an old key was exposed in a screenshot. Do not paste the key into a normal code cell.


In [ ]:
%pip -q install openai scikit-learn sentence-transformers pandas

In [ ]:
import os, json, csv, time, re
from getpass import getpass
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Hidden input: the key exists only in this Colab runtime memory.
OPENROUTER_API_KEY = getpass('Paste your OpenRouter API key (input hidden): ').strip()

if not OPENROUTER_API_KEY:
    raise ValueError('No OpenRouter API key was entered.')
if '\\n' in OPENROUTER_API_KEY or '\\r' in OPENROUTER_API_KEY:
    raise ValueError('The API key contains a newline. Paste one key only.')

print('✅ API key loaded for this runtime only. It will not be displayed or saved in the notebook.')


## 1. Fixed data and classifier baseline

In [ ]:
QUESTIONS = [('My forearms are always really sore after playing badminton.', 'technique'), ('When hitting the ball, it will hit the rim.', 'technique'), ("Why can't I suppress the smash", 'technique'), ("Why can't I hit my forehand high clear", 'technique'), ('The ball always goes out of bounds when hit to the right rear.', 'technique'), ('Besides lob shots, how else can you hit a shot at the net?', 'technique'), ('The drop shot either went too high or hit the net.', 'technique'), ('How to correct a tight grip and stiff arm when playing', 'technique'), ('How to practice switching to a backhand grip after hitting a forehand?', 'technique'), ('Backhand hitting technique practice', 'technique'), ('my forehand clear cannot travel far even with big swings', 'technique'), ('My backhand serve often goes into the net.', 'technique'), ('My net play is terrible, making me easy for my opponent to attack.', 'technique'), ('My defensive lob is too short, making it easy for the opponent to spike it.', 'technique'), ('I often push the ball too far at the net, causing it to go out of bounds.', 'technique'), ('What can I do if my footwork can never catch up with the shuttle as a beginner?', 'footwork'), ('I keep failing to do the split‑step in badminton. Are there any drills I can practice?', 'footwork'), ('Why do I often lose my balance after hitting the shuttle and have poor weight transfer?', 'footwork'), ('I tend to overshoot and cannot stop when lunging to the net. How can I fix this?', 'footwork'), ('My backcourt footwork is too slow and I can barely reach the shuttle. How should I train?', 'footwork'), ('I drag my feet and cannot take wide steps when moving sideways. What is the cause?', 'footwork'), ('I always forget to return to the center base after one shot. How do I build this habit?', 'footwork'), ('What is the difference between a hop step and a split‑step, and when should I use each?', 'footwork'), ('My feet drag on the court and I take too many tiny shuffling steps. How to correct it?', 'footwork'), ('My movement is always a split second late when defending drop shots. What tips can help?', 'footwork'), ('When should I use side steps versus cross‑steps? I often pick the wrong one.', 'footwork'), ('I stay too upright and run on my toes. How do I fix my body posture for footwork?', 'footwork'), ('I struggle to get back to the net quickly from the rear court. What footwork pattern should I use?', 'footwork'), ('My ready stance is stiff and I do not know how to adjust my position flexibly.', 'footwork'), ('It is hard for me to speed up again right after an emergency stop. What footwork mistake am I making?', 'footwork'), ('How should beginners choose a suitable badminton racket and what key parameters should I check?', 'equipment'), ('What string tension is good for new players? What problems will high tension cause?', 'equipment'), ('What is the difference between badminton shoes and regular sneakers? Can I play in running shoes?', 'equipment'), ('How often should I replace my grip? What can I do when my grip gets slippery?', 'equipment'), ('The racket handle feels too thick or too thin. How can I adjust it?', 'equipment'), ('How can I tell if a badminton shuttlecock is genuine? Is there a big difference between different‑priced shuttles?', 'equipment'), ('Do beginners need to buy an expensive professional racket right away?', 'equipment'), ('When should I wear wrist or knee supports? Will protective gear limit my movements?', 'equipment'), ('Do I have to cut out the whole string once it breaks? Can I repair just part of it?', 'equipment'), ('How do I tell apart offensive and defensive rackets. Which one suits beginners better?', 'equipment'), ('Is a grip the same as an overgrip? Should I pick thin or thick grips?', 'equipment'), ('My feet keep slipping on the court. Is it caused by my shoes or the court surface?', 'equipment'), ('Heavy rackets are hard to swing while light ones lack power. How do I pick the right racket weight?', 'equipment'), ('Do I have to use a dedicated badminton bag or can I carry rackets in an ordinary backpack?', 'equipment'), ('Shuttlecocks break feathers easily in cold weather. Are there any more durable shuttlecocks?', 'equipment'), ('How can I fix my wrist flick on the forehand clear?', 'technique'), ('What is the correct arm movement for a powerful smash?', 'technique'), ('Why does my backhand drop shot always land too high?', 'technique'), ('How should I rotate my shoulders when hitting overhead shots?', 'technique'), ('What mistakes do beginners usually make with grip pressure?', 'technique'), ('How do I perform a reliable net tap at the front court?', 'technique'), ('How can I make my cross‑court drive more accurate?', 'technique'), ('Why is my serve always going out of the back boundary?', 'technique'), ('What is the proper hitting point for a forehand lift?', 'technique'), ('How to improve the consistency of my backhand clear?', 'technique'), ('What steps should I use to quickly reach wide forehand corners?', 'footwork'), ('How do I recover to the centre after moving to the back court?', 'footwork'), ('Why do I often lose balance when doing side shuffles?', 'footwork'), ('Which foot moves first when I chase a low net ball?', 'footwork'), ('How can I shorten my reaction time for diagonal movement?', 'footwork'), ('What is the correct split‑step timing before each shot?', 'footwork'), ('How to adjust my steps when moving backwards for overhead shots?', 'footwork'), ('Why do my feet get tangled during fast lateral movement?', 'footwork'), ('What drills help beginners build faster court mobility?', 'footwork'), ('How should I position my weight before moving to the front?', 'footwork'), ('What weight racket is most suitable for new intermediate players?', 'equipment'), ('How tight should my badminton strings be strung for casual training?', 'equipment'), ('Are thick‑soled shoes better for quick side‑to‑side movement?', 'equipment'), ('What differences between synthetic and natural feather shuttlecocks?', 'equipment'), ('Do overgrips help to reduce hand slippage during long matches?', 'equipment'), ('Which racket frame material offers more power for beginners?', 'equipment'), ('How often should I replace my worn badminton shoe insoles?', 'equipment'), ('Is a heavier grip helpful when practising powerful smashes?', 'equipment'), ('What protective gear can prevent ankle injuries on slippery courts?', 'equipment'), ('How do I choose suitable badminton socks for long training sessions?', 'equipment')]

In [ ]:
texts=[q for q,_ in QUESTIONS]; labels=[y for _,y in QUESTIONS]
X_train,X_test,y_train,y_test=train_test_split(texts,labels,test_size=0.35,random_state=0,stratify=labels)
majority=Counter(y_test).most_common(1)[0][0]
baseline_acc=accuracy_score(y_test,[majority]*len(y_test))
clf=make_pipeline(TfidfVectorizer(ngram_range=(1,1)),LogisticRegression(max_iter=1000))
clf.fit(X_train,y_train)
t0=time.perf_counter(); pred=clf.predict(X_test); latency=(time.perf_counter()-t0)/len(X_test)*1000
acc=accuracy_score(y_test,pred)
print(f'Total: {len(QUESTIONS)} | train: {len(X_train)} | test: {len(X_test)} | test support: {dict(Counter(y_test))}')
print(f'Majority baseline: {baseline_acc:.1%}')
print(f'TF-IDF accuracy: {acc:.1%} ({sum(a==b for a,b in zip(pred,y_test))}/{len(y_test)})')
print(f'TF-IDF latency this run: {latency:.3f} ms/item')

## 2. Knowledge base and retrieval

In [ ]:
NOTES = [
  {
    "id": "doc0",
    "category": "technique",
    "text": "For beginners, the correct grip pressure for badminton is similar to holding a bird gently. Squeezing too tight causes wrist stiffness and reduces shot control."
  },
  {
    "id": "doc1",
    "category": "footwork",
    "text": "The standard ready stance for badminton footwork: feet shoulder-width apart, knees slightly bent, weight on the balls of the feet, racket held at chest height."
  },
  {
    "id": "doc2",
    "category": "footwork",
    "text": "Split step timing: perform the small hop just before your opponent makes contact with the shuttle, not after they hit it. This improves reaction speed."
  },
  {
    "id": "doc3",
    "category": "technique",
    "text": "Beginner backcourt forehand clear uses a full shoulder rotation, not just arm strength. Hit the shuttle at the highest point in front of your body."
  },
  {
    "id": "doc4",
    "category": "equipment",
    "text": "For new players, recommended racket weight is 85-90 grams. Heavier rackets generate more power but cause faster arm fatigue during long sessions."
  },
  {
    "id": "doc5",
    "category": "equipment",
    "text": "Beginner string tension should be 20-22 lbs. Higher tension gives better accuracy but reduces forgiveness and is more likely to cause wrist soreness."
  },
  {
    "id": "doc6",
    "category": "equipment",
    "text": "Badminton shoes have non-marking gum rubber soles designed for grip on indoor courts. Running shoes are unsuitable because they lack lateral support and can slip."
  },
  {
    "id": "doc7",
    "category": "technique",
    "text": "To fix a net shot that hits the tape: reduce wrist power, contact the shuttle at a higher point, and brush the shuttle cork gently instead of pushing it."
  },
  {
    "id": "doc8",
    "category": "footwork",
    "text": "When moving to the forehand rear corner, use a cross-over step first, then adjust with side steps. Do not run backwards facing the net."
  },
  {
    "id": "doc9",
    "category": "footwork",
    "text": "After every shot, return to the center of the court (T-junction area) to maintain balanced coverage for your opponent's next return."
  },
  {
    "id": "doc10",
    "category": "equipment",
    "text": "overgrips should be replaced after 10-15 hours of play. Signs of wear include slipperiness, visible fraying and loss of original texture."
  },
  {
    "id": "doc11",
    "category": "technique",
    "text": "For defensive lifts, contact the shuttle low and use a relaxed wrist flick. Hitting it too high and short gives your opponent an easy smash opportunity."
  },
  {
    "id": "doc12",
    "category": "footwork",
    "text": "Side shuffle steps are used for wide side-to-side movement. Cross steps are for fast long-distance movement to rear corners."
  },
  {
    "id": "doc13",
    "category": "equipment",
    "text": "Feather shuttlecocks break faster in cold dry air. Humid conditions make feathers softer and more durable but slower in flight."
  },
  {
    "id": "doc14",
    "category": "technique",
    "text": "Backhand grip change: rotate the racket with your thumb and index finger, do not loosen your whole hand and re-grip from scratch."
  },
  {
    "id": "doc15",
    "category": "footwork",
    "text": "Lunging to the net: land on your heel first, then roll to the forefoot. Keep your knee aligned with your toes to avoid joint injury."
  },
  {
    "id": "doc16",
    "category": "equipment",
    "text": "Beginners should start with medium-flex rackets. Stiff rackets require fast swing speed to generate power and are harder to control."
  },
  {
    "id": "doc17",
    "category": "equipment",
    "text": "Wrist supports are not recommended for regular beginner training. They limit natural wrist movement and weaken your own wrist strength development."
  },
  {
    "id": "doc18",
    "category": "technique",
    "text": "The main cause of shuttle going out on forehand clears is hitting the shuttle too late, behind your shoulder line. Move earlier to hit in front."
  },
  {
    "id": "doc19",
    "category": "footwork",
    "text": "To improve footwork speed, practice shadow footwork drills for 15 minutes per session. Focus on rhythm rather than maximum speed at first."
  }
]

In [ ]:
from sentence_transformers import SentenceTransformer
embedder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
note_matrix=embedder.encode([n['text'] for n in NOTES],normalize_embeddings=True)

def retrieve(query, category, top_k=3):
    q=embedder.encode([query],normalize_embeddings=True)[0]
    scores=note_matrix @ q
    hits=[]
    for i,n in enumerate(NOTES):
        if n['category']==category:
            hits.append((float(scores[i]),n))
    hits.sort(key=lambda x:x[0],reverse=True)
    return [{'score':s,**n} for s,n in hits[:top_k]]
print('Retrieval ready: MiniLM + category-filtered top-3.')

## 3. Grounded advisor

In [ ]:
from openai import OpenAI
client=OpenAI(base_url='https://openrouter.ai/api/v1',api_key=OPENROUTER_API_KEY)
MODEL='openai/gpt-4o-mini'
SYSTEM='''You are a beginner badminton training assistant.
Use ONLY the supplied NOTES. Do not add facts, timings, repetitions, diagnoses, or safety claims that are not present in the notes.
Return ONLY valid JSON with keys: category, likely_issue, drill, evidence_note_ids.
The drill must be specific and actionable. If the notes do not support a useful answer, set likely_issue and drill to "The notes do not say" and use an empty evidence_note_ids list.'''

def call_advisor(question,category,hits):
    notes='\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    user=f'USER QUESTION: {question}\nROUTED CATEGORY: {category}\n\nNOTES:\n{notes}'
    r=client.chat.completions.create(model=MODEL,messages=[{'role':'system','content':SYSTEM},{'role':'user','content':user}],temperature=0,max_tokens=220)
    text=r.choices[0].message.content.strip()
    m=re.search(r'\{.*\}',text,re.S)
    obj=json.loads(m.group(0) if m else text)
    usage=getattr(r,'usage',None)
    return obj, {'in':getattr(usage,'prompt_tokens',0) or 0,'out':getattr(usage,'completion_tokens',0) or 0}
print('Advisor ready:',MODEL)

## 4. Ten fixed end-to-end advice cases

In [ ]:
CASES = [
  {
    "case_id": "A01",
    "input": "My grip is so tight that my wrist becomes stiff and my shots feel hard to control.",
    "expected_category": "technique",
    "required_evidence": "doc0",
    "expected_advice": "relax grip pressure; avoid squeezing too tightly"
  },
  {
    "case_id": "A02",
    "input": "I react late because I only move after my opponent has already hit the shuttle.",
    "expected_category": "footwork",
    "required_evidence": "doc2",
    "expected_advice": "perform the split step just before opponent contact"
  },
  {
    "case_id": "A03",
    "input": "My forehand clear lacks distance even though I swing hard with my arm.",
    "expected_category": "technique",
    "required_evidence": "doc3",
    "expected_advice": "use full shoulder rotation and contact high in front"
  },
  {
    "case_id": "A04",
    "input": "I am a beginner and my racket makes my arm tired quickly. What weight should I look for?",
    "expected_category": "equipment",
    "required_evidence": "doc4",
    "expected_advice": "recommend 85-90 g for new players"
  },
  {
    "case_id": "A05",
    "input": "High string tension makes my wrist sore. What beginner setting should I use?",
    "expected_category": "equipment",
    "required_evidence": "doc5",
    "expected_advice": "recommend 20-22 lbs and explain high-tension trade-off"
  },
  {
    "case_id": "A06",
    "input": "Can I use my running shoes for badminton even though I slide during side movements?",
    "expected_category": "equipment",
    "required_evidence": "doc6",
    "expected_advice": "use badminton shoes with lateral support and gum rubber soles"
  },
  {
    "case_id": "A07",
    "input": "My net shot keeps touching the tape because I push too hard. What should I change?",
    "expected_category": "technique",
    "required_evidence": "doc7",
    "expected_advice": "reduce wrist power; contact higher; brush cork gently"
  },
  {
    "case_id": "A08",
    "input": "I am slow reaching the forehand rear corner and often run backwards.",
    "expected_category": "footwork",
    "required_evidence": "doc8",
    "expected_advice": "cross-over step first, then side steps; do not run backwards facing net"
  },
  {
    "case_id": "A09",
    "input": "After every shot I stay where I hit it and then cannot cover the next reply.",
    "expected_category": "footwork",
    "required_evidence": "doc9",
    "expected_advice": "return to centre/T-junction after each shot"
  },
  {
    "case_id": "A10",
    "input": "I want a simple drill to improve court movement speed without losing rhythm.",
    "expected_category": "footwork",
    "required_evidence": "doc19",
    "expected_advice": "shadow footwork for 15 minutes, prioritising rhythm before max speed"
  }
]

In [ ]:
rows=[]; token_in=token_out=0
for i,c in enumerate(CASES,1):
    probs=clf.predict_proba([c['input']])[0]
    classes=clf.named_steps['logisticregression'].classes_
    cat=str(clf.predict([c['input']])[0])
    confidence=float(max(probs))
    hits=retrieve(c['input'],cat,top_k=3)
    answer,usage=call_advisor(c['input'],cat,hits)
    token_in += usage['in']; token_out += usage['out']
    retrieved_ids=[h['id'] for h in hits]
    drill=str(answer.get('drill','')) if isinstance(answer,dict) else ''
    row={**c,
         'model_category':cat,
         'category_correct_0_1':int(cat==c['expected_category']),
         'classifier_confidence':round(confidence,4),
         'retrieved_note_ids':';'.join(retrieved_ids),
         'evidence_hit_0_1':int(c['required_evidence'] in retrieved_ids),
         'retrieved_notes':' | '.join(h['text'] for h in hits),
         'model_advice':json.dumps(answer,ensure_ascii=False),
         'prompt_tokens':usage['in'],
         'completion_tokens':usage['out'],
         'abstained_0_1':int('notes do not say' in drill.lower())}
    rows.append(row)
    print(f"[{i}/10] {c['case_id']} | route={cat} expected={c['expected_category']} conf={confidence:.3f} | notes={row['retrieved_note_ids']} | evidence_hit={row['evidence_hit_0_1']}")
    print(json.dumps(answer,ensure_ascii=False,indent=2),'\n')

df=pd.DataFrame(rows)
df.to_csv('advice_eval_outputs.csv',index=False)
auto_summary={
    'route_correct':int(df.category_correct_0_1.sum()),
    'evidence_hits_at_3':int(df.evidence_hit_0_1.sum()),
    'abstentions':int(df.abstained_0_1.sum()),
    'prompt_tokens':int(token_in),
    'completion_tokens':int(token_out),
}
print('Saved advice_eval_outputs.csv')
print('AUTOMATIC DIAGNOSTICS')
print(f"Route correctness: {auto_summary['route_correct']}/10")
print(f"Evidence recall@3: {auto_summary['evidence_hits_at_3']}/10")
print(f"Abstentions: {auto_summary['abstentions']}/10")
print('Tokens:',token_in,'input /',token_out,'output')


## 5. HAND MARK — required by instructor

Read each output beside its retrieved notes. Enter one row per case below.

- **Specific = 1** only if the drill gives a concrete action, not generic encouragement.
- **Supported = 1** only if the factual claims/drill are supported by the retrieved notes.
- **No invention = 1** only if it adds no unsupported timing, repetitions, diagnosis, safety claim or factual detail.

This must be your judgement; do not replace it with an LLM judge.

In [ ]:
# Fill all ten rows after reading the outputs above.
# Format: 'A01': (specific, supported, no_invention, 'short note')
MANUAL_MARKS = {
    'A01': (None,None,None,''),
    'A02': (None,None,None,''),
    'A03': (None,None,None,''),
    'A04': (None,None,None,''),
    'A05': (None,None,None,''),
    'A06': (None,None,None,''),
    'A07': (None,None,None,''),
    'A08': (None,None,None,''),
    'A09': (None,None,None,''),
    'A10': (None,None,None,''),
}
MANUAL_MARKS

In [ ]:
assert all(all(v is not None for v in MANUAL_MARKS[c][:3]) for c in MANUAL_MARKS), 'Complete the 0/1 marks above first.'
mark_rows=[]
for r in rows:
    s,sup,noinv,note=MANUAL_MARKS[r['case_id']]
    mark_rows.append({**r,'specific_0_1':s,'supported_0_1':sup,'no_invention_0_1':noinv,'marker_notes':note})
final=pd.DataFrame(mark_rows)
final.to_csv('advice_eval_final.csv',index=False)
manual_summary={
    'specific':sum(final.specific_0_1)/len(final),
    'supported':sum(final.supported_0_1)/len(final),
    'no_invention':sum(final.no_invention_0_1)/len(final),
}
final_summary={**auto_summary,
    'specific_passes':int(final.specific_0_1.sum()),
    'supported_passes':int(final.supported_0_1.sum()),
    'no_invention_passes':int(final.no_invention_0_1.sum()),
}
with open('final_eval_summary.json','w',encoding='utf-8') as f:
    json.dump(final_summary,f,indent=2)
print('FINAL HUMAN-MARKED ADVICE METRICS')
for k,v in manual_summary.items(): print(f'{k}: {v:.0%} ({int(round(v*10))}/10)')
print('\nAutomatic diagnostics')
print(f"route: {auto_summary['route_correct']}/10 | evidence recall@3: {auto_summary['evidence_hits_at_3']}/10 | abstentions: {auto_summary['abstentions']}/10")
print(f"tokens: {auto_summary['prompt_tokens']} input / {auto_summary['completion_tokens']} output")
print('Saved advice_eval_final.csv and final_eval_summary.json')
